# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset describes mass spectrometry-based protein abundance, modifications, and sequences in human samples.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("\n=== DATASET OVERVIEW ===")
print(f"Title   : {getattr(metadata, 'name', '')}")
print(f"Version : {getattr(metadata, 'version', '')}")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print("\nDescription:")
print(getattr(metadata, 'description', ''))

## 2. Data Overview
Review all available record sets and their `@id`s, and inspect their fields (column names and corresponding `@id`s).

<sub>**Note:** In a Croissant dataset, each record set and its columns/fields have unique `@id`s, which we use as references for querying data and extracting subsets.</sub>

In [ ]:
# Gather all record sets
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in metadata. Attempting to retrieve from the Dataset directly.")
    try:
        record_sets = dataset.metadata._data.get('recordSet', [])
    except Exception as e:
        print(f"Error retrieving record sets: {e}")

# List record set IDs
print("--- Available Record Sets ---")
record_set_ids = []
for rs in record_sets:
    # Each could be dict or string, robust extraction:
    if isinstance(rs, dict) and '@id' in rs:
        rec_id = rs['@id']
    elif isinstance(rs, str):
        rec_id = rs
    else:
        continue
    record_set_ids.append(rec_id)
    print(f"@id: {rec_id}")
    # Try to get columns/fields for each record set
    rs_metadata = None
    # Approach: scan through metadata JSON for exact recordSet entity
    for rs_candidate in dataset.metadata._data.get('recordSet', []):
        if isinstance(rs_candidate, dict) and rs_candidate.get('@id','') == rec_id:
            rs_metadata = rs_candidate
            break
    if rs_metadata:
        print("  Fields (columns):")
        for fld in rs_metadata.get('field', []):
            if isinstance(fld, dict):
                print(f"      - {fld.get('@id', str(fld))}")
            else:
                print(f"      - {fld}")
    else:
        print("  Fields metadata not found.")

if len(record_set_ids) == 0:
    print("No record sets with fields available. Please check the Croissant definition or contact provider.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Here, we will demonstrate loading the **main protein table** (the largest/primary tabular record set) into a pandas DataFrame.

**All entities are always referenced by their `@id` values.**

In [ ]:
# If you know the main record set, put its @id here:
# Example (ID is illustrative, update after inspecting record_set_ids):
if record_set_ids:
    main_record_set_id = record_set_ids[0]  # <- Set to your record set's @id
    print(f"Using record set: {main_record_set_id}")
else:
    raise ValueError("No record sets found.")

# List fields for this record set
print(f"\nFields in {main_record_set_id}:")
main_field_ids = []
main_fields = []
for rs in dataset.metadata._data.get('recordSet', []):
    if rs.get('@id','') == main_record_set_id:
        for fld in rs.get('field', []):
            if isinstance(fld, dict):
                main_field_ids.append(fld.get('@id', str(fld)))
                main_fields.append(fld)
            else:
                main_field_ids.append(fld)
        break
print(main_field_ids)

# Extract all records from the chosen record set
print("\nLoading records...")
dataframe = pd.DataFrame(dataset.records(record_set=main_record_set_id))
print("Columns:")
print(dataframe.columns.tolist())
dataframe.head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps, such as:
* Filtering records based on specific criteria (for example, protein abundance above a threshold),
* Normalizing numeric fields,
* Grouping/categorizing data.

This is only an example — modify numeric_field/group_field IDs according to your record set.

All fields referenced by their `@id`!

In [ ]:
# Choose a numeric field to analyze (set to your actual field @id)
possible_numeric_fields = [col for col in dataframe.columns if dataframe[col].dtype in ['int64', 'float64']]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # As fallback, try to guess from column names
    numeric_field_id = dataframe.columns[0]
print(f"Numeric field selected: {numeric_field_id}")

threshold = dataframe[numeric_field_id].mean() # For demonstration, can set as needed
# Filtering records with value above the mean
filtered_df = dataframe[dataframe[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records\n")
print(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Demonstrate grouping by another field if available
possible_group_fields = [col for col in dataframe.columns if dataframe[col].dtype == 'object' and col != numeric_field_id]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    print(f"\nGrouping data by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
    print(grouped_df.head())
else:
    print("No string/categorical field available for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

All plots below use fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(dataframe[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by a group/category field if possible
if possible_group_fields:
    plt.figure(figsize=(12,5))
    top_categories = dataframe[group_field_id].value_counts().nlargest(10).index
    sns.boxplot(
        data=dataframe[dataframe[group_field_id].isin(top_categories)], 
        x=group_field_id, y=numeric_field_id
    )
    plt.title(f'{numeric_field_id} by {group_field_id} (top 10 categories)')
    plt.xticks(rotation=30)
    plt.show()
else:
    print("No categorical field for groupwise boxplot.")

## 6. Conclusion
In this notebook, we have:
* Loaded and reviewed metadata fields from the FAIR² protein mass spectrometry dataset, referencing all entities by their Croissant `@id`
* Explored the available record sets and fields, loaded the main record set into pandas,
* Applied common EDA steps using field `@id` references,
* Visualized data distributions and relationships.

Further analysis, such as advanced statistical analysis or machine learning, can be performed using this framework. See the [mlcroissant documentation](https://github.com/mlcommons/croissant) for more features.